# Search: try the cascade yourself

The free-play notebook: draw a box on any image and see what the cascade finds. No metrics are computed here, so it opens and runs quickly.

**Sections**
1. **Setup:** imports, settings and the DINOv2 model.
2. **Your test set:** loaded from `annotations.json` (made with `notebooks/labeling.ipynb`).
3. **Search the test set:** results are boxes from your test set, with ✓/✗ against the label you expect.
4. **Search a new gallery:** any folder, glob or list of photos, no labels needed; you judge the top-n yourself.

Related notebooks:
- `notebooks/labeling.ipynb` builds the test set.
- `notebooks/evaluation.ipynb` measures and compares approaches on it.

Everything needs JupyterLab with ipympl (`uv run jupyter lab`).

## 1. Setup

This cell switches matplotlib to interactive widgets (needed to draw boxes), imports the package, sets the main knobs and loads DINOv2 once for both search apps:

- `TESTSET_DIR`: your labelled test set (default `data/testset/`, or the `VS_TESTSET` environment variable).
- `DINO_MODEL` / `DINO_RESOLUTION`: the stage-1 model (448 px default, 224 px is about 3× faster).
- `RETRIEVAL_K`: candidates kept by stage 1. `SPARSE_K`: candidates re-checked by stage 2 (keypoint matching).
- `MASKER`: how your box becomes a mask. `BoxMasker()` uses the box itself; SAM3 can replace it once it passes its check (`scripts/try_sam3.py`):
  ```python
  from visual_search.masking.sam3 import Sam3Config, Sam3Masker
  MASKER = Sam3Masker(Sam3Config(model_dir=ROOT / "models/sam3", variant="fp16"))
  ```
- `CACHE_DIR`: computed features are stored here, so later runs are fast.

In [ ]:
%matplotlib widget
import logging
import os
from pathlib import Path

from visual_search.cascade import Cascade
from visual_search.data import AnnotationSet, load_gallery, make_synthetic_dataset, split_dataset
from visual_search.data.schema import parse_dataset
from visual_search.device import device_sync
from visual_search.masking import BoxMasker
from visual_search.models.dinov2 import Dinov2Backbone
from visual_search.retrieval import DinoRetriever, GroundedDinoRetriever
from visual_search.stages import SparseVerificationStage
from visual_search.ui import SearchApp

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
TESTSET_DIR = Path(os.environ.get("VS_TESTSET", ROOT / "data" / "testset"))
CACHE_DIR = ROOT / "cache"

DINO_MODEL = "facebook/dinov2-small"
DINO_RESOLUTION = 448
RETRIEVAL_K = 100
SPARSE_K = 20
MASKER = BoxMasker()

backbone = Dinov2Backbone(DINO_MODEL, resolution=DINO_RESOLUTION)
sync = device_sync(backbone.device)
print("DINOv2 on", backbone.device)

## 2. Your test set

This cell loads the labelled boxes from `annotations.json`. Every box is one item; images you marked *reviewed* without boxes are distractors.

Without a test set it falls back to generated synthetic data (smoke test only), so the apps still work.

In [ ]:
if (TESTSET_DIR / "annotations.json").exists():
    testset = AnnotationSet.load(TESTSET_DIR)
    print(testset.summary())
    testset_items = parse_dataset(testset.to_records())
    search_gallery, search_queries = testset_items, [it for it in testset_items if it.label]
else:
    print(f"No annotations.json in {TESTSET_DIR}; using synthetic data.")
    synthetic = split_dataset(make_synthetic_dataset(n_instances=12, views_per_instance=3, n_distractors=40, seed=0), seed=0)
    search_gallery, search_queries = synthetic.gallery, synthetic.queries
print(f"{len(search_gallery)} gallery items, {len(search_queries)} labelled boxes to pick as queries")

## 3. Search the test set

This cell builds a search app whose **gallery is every box in your test set**, plus the distractor images.

- **Pipeline:** DINOv2 (`crop_cls`: your box is cropped and described) retrieves the top `RETRIEVAL_K`, then SIFT keypoint verification re-checks the top `SPARSE_K`.
- **First run:** indexes all boxes, which takes a while once; after that everything comes from `cache/`.

In [ ]:
search_cascade = Cascade(
    DinoRetriever(backbone, pooling="crop_cls", cache_dir=CACHE_DIR),
    retrieval_k=RETRIEVAL_K,
    stages=[SparseVerificationStage("sift", top_k=SPARSE_K, model_type="H", method="magsac", cache_dir=CACHE_DIR)],
    sync=sync,
)
testset_app = SearchApp(search_cascade, gallery=search_gallery, queries=search_queries, masker=MASKER, top_k=5)
testset_app.widget

**How to use it**

1. **Pick a query.** Choose a labelled box from the **Query** dropdown (`image#n [label]`), or type the path of any image and press **Load path**.
2. **Draw or adjust the box.** *Box* mode: drag a rectangle. *Points* mode: left click = on the object, right click or shift+click = exclude. The blue overlay previews the mask.
3. **Check the Expected label.**
   - It's filled in automatically for a test-set box.
   - For a new image, type or pick the label of the object you drew (it autocompletes from the test set) to get ✓/✗. Leave it empty for just the ranking.
4. **Press Search.** You get one row per stage with the top 5:
   - each result is a crop around its box (yellow), with its score, label and a green ✓ / red ✗ frame;
   - a query picked from the test set never finds its own box.
5. **Pick a result in Detail** to see the keypoint matches (green = inliers) and the gallery box warped onto your query.

`testset_app.last_result` holds all rankings and score components after a search.

## 4. Search a new gallery

Search any collection of photos for your object: no labels, no boxes, no test set needed.

**What happens under the hood (gallery grounding)**
- Photos usually show whole scenes, so the object has to be **found inside each photo** first.
- DINOv2 describes every gallery photo as a 32×32 grid of patch features, computed once and cached.
- For a query, the features under your box are averaged and compared with every patch of every photo. Where they are similar, a **region** (box) is drawn; up to `MAX_REGIONS` per photo.
- **Stage 1** ranks these regions, so a photo can appear more than once. **Stage 2** checks the top regions with keypoint matching *inside the region*.

**Settings in the next cell**
- `GALLERY_SOURCE` can be:
  - a folder (all images in it and its subfolders);
  - a glob pattern such as `"/Users/me/Pictures/showroom/**/*.jpg"`;
  - a Python list of file paths.
- `TOP_N` is how many results to show.

The default searches everything under `data/`.

In [ ]:
GALLERY_SOURCE = ROOT / "data"  # a folder, a glob pattern string, or a list of paths
TOP_N = 10
MAX_REGIONS = 3  # regions (results) per gallery photo

new_gallery = parse_dataset(load_gallery(GALLERY_SOURCE))
print(f"{len(new_gallery)} gallery images from {GALLERY_SOURCE}")

This cell builds the search app for that gallery.

- **Stage 1:** `GroundedDinoRetriever` finds and scores the regions.
- **Stage 2:** SIFT verification re-ranks the top `max(SPARSE_K, TOP_N)` regions.
- **First run:** computes DINOv2 features for every gallery photo (about 55 ms per photo on an M2); later runs read them from `cache/`.
- **No ✓/✗:** there is no *Expected* field; you judge the results yourself.
- **Queries:** the dropdown offers your test-set boxes for convenience. For a new photo, use **Load path**.

In [ ]:
grounded_cascade = Cascade(
    GroundedDinoRetriever(backbone, cache_dir=CACHE_DIR, max_regions=MAX_REGIONS),
    retrieval_k=RETRIEVAL_K,
    stages=[SparseVerificationStage("sift", top_k=max(SPARSE_K, TOP_N), cache_dir=CACHE_DIR)],
    sync=sync,
)
gallery_app = SearchApp(grounded_cascade, gallery=new_gallery, queries=search_queries, masker=MASKER,
                        top_k=TOP_N, show_expected_label=False)
gallery_app.widget

**How to use it**

1. **Load a query** with **Load path** (or pick a test-set box) and draw a box around the object.
2. **Press Search.** Each result is a crop around the **region** where the object was found (yellow box):
   - **Stage-1 row:** the score is the cosine similarity between your object and the region (higher = more similar). `peak` is the best single-patch similarity.
   - **Stage-2 row:** the score is `inliers + cosine`. Above 8 means the keypoint check confirmed the same physical surface.
3. **Pick a result in Detail** to see the keypoint matches inside the region. The Detail list shows each region's coordinates.

**Tips**
- **Region size:** `GroundedDinoRetriever(..., delta=0.1)` controls it. Smaller = tighter regions around the strongest spot.
- **Memory:** all patch grids are kept in RAM, about 0.75 MB per photo at 448 px.

`gallery_app.last_result` holds every candidate with its `region` after a search.